# 07 - Feature Engineering

## Project: ECOMMERCE-AI-PLATFORM

This notebook creates:

1. Customer-level features
2. Product-level features
3. Seller-level features
4. Order-level features
5. Monthly sales features
6. Category-level features

The engineered outputs will be used for:

- Customer segmentation
- Customer lifetime value
- Customer churn prediction
- Product recommendation
- Sales forecasting
- AI-generated business insights
- Future Power BI dashboards

In [1]:
import os
import warnings
from getpass import getpass

import pandas as pd
import numpy as np

from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
PROJECT_PATH = r"C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM"

PROCESSED_DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

FEATURE_ENGINEERED_PATH = os.path.join(
    PROCESSED_DATA_PATH,
    "feature_engineered"
)

os.makedirs(FEATURE_ENGINEERED_PATH, exist_ok=True)

print("Project path:")
print(PROJECT_PATH)

print("\nFeature-engineered output path:")
print(FEATURE_ENGINEERED_PATH)

Project path:
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM

Feature-engineered output path:
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\feature_engineered


In [3]:
if os.path.exists(PROJECT_PATH):
    print("Project directory exists.")
else:
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_PATH}"
    )

if os.path.exists(PROCESSED_DATA_PATH):
    print("Processed data directory exists.")
else:
    raise FileNotFoundError(
        f"Processed data directory not found: {PROCESSED_DATA_PATH}"
    )

print("Path verification completed successfully.")

Project directory exists.
Processed data directory exists.
Path verification completed successfully.


In [4]:
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "ecommerce_ai_db"
DB_USER = "postgres"

DB_PASSWORD = getpass(
    "Enter your PostgreSQL password privately: "
)

print("Database configuration loaded.")
print(f"Host: {DB_HOST}")
print(f"Port: {DB_PORT}")
print(f"Database: {DB_NAME}")
print(f"User: {DB_USER}")
print("Password: Hidden")

Enter your PostgreSQL password privately:  ········


Database configuration loaded.
Host: localhost
Port: 5432
Database: ecommerce_ai_db
User: postgres
Password: Hidden


In [5]:
DATABASE_URL = (
    f"postgresql+psycopg2://"
    f"{DB_USER}:{DB_PASSWORD}@"
    f"{DB_HOST}:{DB_PORT}/"
    f"{DB_NAME}"
)

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True
)

print("PostgreSQL engine created successfully.")

PostgreSQL engine created successfully.


In [6]:
try:

    with engine.connect() as connection:

        result = connection.execute(
            text(
                """
                SELECT
                    current_database() AS database_name,
                    current_user AS user_name;
                """
            )
        )

        connection_info = result.fetchone()

        print("Database connection successful.")
        print(f"Database: {connection_info.database_name}")
        print(f"User: {connection_info.user_name}")

except SQLAlchemyError as error:

    print("Database connection failed.")
    print(error)

Database connection successful.
Database: ecommerce_ai_db
User: postgres


In [7]:
tables_query = """
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tables_df = pd.read_sql(
    text(tables_query),
    engine
)

print("Available PostgreSQL tables:\n")

display(tables_df)

Available PostgreSQL tables:



,table_schema,table_name
0,public,category_translation
1,public,customers
2,public,geolocation
3,public,order_items
4,public,order_payments
5,public,order_reviews
6,public,orders
7,public,products
8,public,sellers


In [8]:
required_tables = {
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "category_translation"
}

available_tables = set(
    tables_df["table_name"]
)

missing_tables = (
    required_tables
    - available_tables
)

if missing_tables:

    print("Missing required tables:")
    print(missing_tables)

else:

    print(
        "All required Olist tables are available."
    )

All required Olist tables are available.


In [9]:
schema_query = """
SELECT
    table_name,
    column_name,
    data_type,
    udt_name,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY
    table_name,
    ordinal_position;
"""

schema_df = pd.read_sql(
    text(schema_query),
    engine
)

display(schema_df)

,table_name,column_name,data_type,udt_name,is_nullable
0,category_translation,product_category_name,text,text,YES
1,category_translation,product_category_name_english,text,text,YES
2,customers,customer_id,text,text,YES
3,customers,customer_unique_id,text,text,YES
4,customers,customer_zip_code_prefix,bigint,int8,YES
...,...,...,...,...,...
56,products,product_category_name_english,text,text,YES
57,sellers,seller_id,text,text,YES
58,sellers,seller_zip_code_prefix,bigint,int8,YES
59,sellers,seller_city,text,text,YES


In [10]:
important_tables = [
    "customers",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "products",
    "sellers"
]

important_schema_df = schema_df[
    schema_df["table_name"].isin(
        important_tables
    )
].copy()

display(
    important_schema_df
)

,table_name,column_name,data_type,udt_name,is_nullable
2,customers,customer_id,text,text,YES
3,customers,customer_unique_id,text,text,YES
4,customers,customer_zip_code_prefix,bigint,int8,YES
5,customers,customer_city,text,text,YES
6,customers,customer_state,text,text,YES
12,order_items,order_id,text,text,YES
13,order_items,order_item_id,bigint,int8,YES
14,order_items,product_id,text,text,YES
15,order_items,seller_id,text,text,YES
16,order_items,shipping_limit_date,text,text,YES


In [11]:
def get_table_schema(
    table_name,
    schema_dataframe
):

    table_schema = schema_dataframe[
        schema_dataframe["table_name"]
        == table_name
    ].copy()

    return table_schema

In [12]:
required_columns = {

    "customers": [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ],

    "orders": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],

    "order_items": [
        "order_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ],

    "order_payments": [
        "order_id",
        "payment_type",
        "payment_installments",
        "payment_value"
    ],

    "order_reviews": [
        "order_id",
        "review_score"
    ],

    "products": [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],

    "sellers": [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
}

for table_name, columns in required_columns.items():

    available_columns = set(
        schema_df.loc[
            schema_df["table_name"] == table_name,
            "column_name"
        ]
    )

    missing_columns = (
        set(columns)
        - available_columns
    )

    if missing_columns:

        print(
            f"{table_name}: Missing columns -> "
            f"{missing_columns}"
        )

    else:

        print(
            f"{table_name}: All required columns available."
        )

customers: All required columns available.
orders: All required columns available.
order_items: All required columns available.
order_payments: All required columns available.
order_reviews: All required columns available.
products: All required columns available.
sellers: All required columns available.


In [13]:
customers_df = pd.read_sql(
    text(
        """
        SELECT *
        FROM public.customers;
        """
    ),
    engine
)

orders_df = pd.read_sql(
    text(
        """
        SELECT *
        FROM public.orders;
        """
    ),
    engine
)

order_items_df = pd.read_sql(
    text(
        """
        SELECT *
        FROM public.order_items;
        """
    ),
    engine
)

order_payments_df = pd.read_sql(
    text(
        """
        SELECT *
        FROM public.order_payments;
        """
    ),
    engine
)

order_reviews_df = pd.read_sql(
    text(
        """
        SELECT *
        FROM public.order_reviews;
        """
    ),
    engine
)

products_df = pd.read_sql(
    text(
        """
        SELECT *
        FROM public.products;
        """
    ),
    engine
)

sellers_df = pd.read_sql(
    text(
        """
        SELECT *
        FROM public.sellers;
        """
    ),
    engine
)

print("Core tables loaded successfully.")

Core tables loaded successfully.


In [14]:
dataset_sizes = pd.DataFrame({

    "dataset": [
        "customers",
        "orders",
        "order_items",
        "order_payments",
        "order_reviews",
        "products",
        "sellers"
    ],

    "rows": [
        len(customers_df),
        len(orders_df),
        len(order_items_df),
        len(order_payments_df),
        len(order_reviews_df),
        len(products_df),
        len(sellers_df)
    ],

    "columns": [
        customers_df.shape[1],
        orders_df.shape[1],
        order_items_df.shape[1],
        order_payments_df.shape[1],
        order_reviews_df.shape[1],
        products_df.shape[1],
        sellers_df.shape[1]
    ]
})

display(dataset_sizes)

,dataset,rows,columns
0,customers,99441,5
1,orders,99441,16
2,order_items,112650,7
3,order_payments,103886,5
4,order_reviews,99224,7
5,products,32951,10
6,sellers,3095,4


In [15]:
date_columns = [

    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:

    if column in orders_df.columns:

        orders_df[column] = pd.to_datetime(
            orders_df[column],
            errors="coerce"
        )

print("Date conversion completed.")

Date conversion completed.


In [16]:
numeric_columns = {

    "order_items": [
        "price",
        "freight_value"
    ],

    "order_payments": [
        "payment_installments",
        "payment_value"
    ],

    "order_reviews": [
        "review_score"
    ],

    "products": [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
}

for column in numeric_columns["order_items"]:

    if column in order_items_df.columns:

        order_items_df[column] = pd.to_numeric(
            order_items_df[column],
            errors="coerce"
        )

for column in numeric_columns["order_payments"]:

    if column in order_payments_df.columns:

        order_payments_df[column] = pd.to_numeric(
            order_payments_df[column],
            errors="coerce"
        )

for column in numeric_columns["order_reviews"]:

    if column in order_reviews_df.columns:

        order_reviews_df[column] = pd.to_numeric(
            order_reviews_df[column],
            errors="coerce"
        )

for column in numeric_columns["products"]:

    if column in products_df.columns:

        products_df[column] = pd.to_numeric(
            products_df[column],
            errors="coerce"
        )

print("Numeric conversion completed.")

Numeric conversion completed.


In [17]:
order_item_features = (

    order_items_df
    .groupby("order_id", as_index=False)
    .agg(

        total_items=(
            "order_item_id",
            "count"
        ),

        total_product_value=(
            "price",
            "sum"
        ),

        total_freight_value=(
            "freight_value",
            "sum"
        )

    )
)

order_item_features[
    "total_order_value"
] = (

    order_item_features[
        "total_product_value"
    ]

    +

    order_item_features[
        "total_freight_value"
    ]
)

display(
    order_item_features.head()
)

,order_id,total_items,total_product_value,total_freight_value,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04


In [18]:
payment_features = (

    order_payments_df
    .groupby("order_id", as_index=False)
    .agg(

        total_payment_value=(
            "payment_value",
            "sum"
        ),

        max_payment_installments=(
            "payment_installments",
            "max"
        ),

        payment_methods_used=(
            "payment_type",
            "nunique"
        )

    )
)

display(
    payment_features.head()
)

,order_id,total_payment_value,max_payment_installments,payment_methods_used
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1


In [19]:
review_features = (

    order_reviews_df
    .groupby("order_id", as_index=False)
    .agg(

        review_score=(
            "review_score",
            "mean"
        )

    )
)

display(
    review_features.head()
)

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


In [20]:
order_features = (

    orders_df

    .merge(
        order_item_features,
        on="order_id",
        how="left"
    )

    .merge(
        payment_features,
        on="order_id",
        how="left"
    )

    .merge(
        review_features,
        on="order_id",
        how="left"
    )
)

order_features[
    "delivery_days"
] = (

    order_features[
        "order_delivered_customer_date"
    ]

    -

    order_features[
        "order_purchase_timestamp"
    ]

).dt.total_seconds() / 86400

order_features[
    "estimated_delivery_days"
] = (

    order_features[
        "order_estimated_delivery_date"
    ]

    -

    order_features[
        "order_purchase_timestamp"
    ]

).dt.total_seconds() / 86400

order_features[
    "delivery_delay_days"
] = (

    order_features[
        "order_delivered_customer_date"
    ]

    -

    order_features[
        "order_estimated_delivery_date"
    ]

).dt.total_seconds() / 86400

display(
    order_features.head()
)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_delivery_days,...,order_weekday,delivery_status,total_items,total_product_value,total_freight_value,total_order_value,total_payment_value,max_payment_installments,payment_methods_used,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,15.544063,...,Monday,On Time,1.0,29.99,8.72,38.71,38.71,1.0,2.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,19.137766,...,Tuesday,On Time,1.0,118.70,22.76,141.46,141.46,1.0,1.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,26.639711,...,Wednesday,On Time,1.0,159.90,19.22,179.12,179.12,3.0,1.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,26.188819,...,Saturday,On Time,1.0,45.00,27.20,72.20,72.20,1.0,1.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,12.112049,...,Tuesday,On Time,1.0,19.90,8.72,28.62,28.62,1.0,1.0,5.0


In [21]:
order_features[
    "is_delivered"
] = (

    order_features[
        "order_status"
    ]

    ==

    "delivered"

).astype(int)

order_features[
    "is_cancelled"
] = (

    order_features[
        "order_status"
    ]

    ==

    "canceled"

).astype(int)

order_features[
    "is_late"
] = (

    order_features[
        "delivery_delay_days"
    ]

    > 0

).astype(int)

order_features[
    "is_on_time"
] = (

    (

        order_features[
            "delivery_delay_days"
        ]

        <= 0

    )

    &

    (

        order_features[
            "is_delivered"
        ]

        == 1

    )

).astype(int)

display(
    order_features[
        [
            "order_id",
            "order_status",
            "delivery_days",
            "delivery_delay_days",
            "is_delivered",
            "is_cancelled",
            "is_late",
            "is_on_time"
        ]
    ].head()
)

,order_id,order_status,delivery_days,delivery_delay_days,is_delivered,is_cancelled,is_late,is_on_time
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,8.436574,-7.107488,1,0,0,1
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,13.782037,-5.355729,1,0,0,1
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,9.394213,-17.245498,1,0,0,1
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,13.208750,-12.980069,1,0,0,1
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2.873877,-9.238171,1,0,0,1


In [22]:
order_features[
    "purchase_year"
] = (

    order_features[
        "order_purchase_timestamp"
    ]

    .dt.year

)

order_features[
    "purchase_month"
] = (

    order_features[
        "order_purchase_timestamp"
    ]

    .dt.month

)

order_features[
    "purchase_quarter"
] = (

    order_features[
        "order_purchase_timestamp"
    ]

    .dt.quarter

)

order_features[
    "purchase_day_of_week"
] = (

    order_features[
        "order_purchase_timestamp"
    ]

    .dt.dayofweek

)

order_features[
    "purchase_hour"
] = (

    order_features[
        "order_purchase_timestamp"
    ]

    .dt.hour

)

order_features[
    "purchase_date"
] = (

    order_features[
        "order_purchase_timestamp"
    ]

    .dt.date

)

order_features[
    "purchase_month_start"
] = (

    order_features[
        "order_purchase_timestamp"
    ]

    .dt.to_period("M")
    .dt.to_timestamp()

)

display(
    order_features.head()
)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_delivery_days,...,is_cancelled,is_late,is_on_time,purchase_year,purchase_month,purchase_quarter,purchase_day_of_week,purchase_hour,purchase_date,purchase_month_start
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,15.544063,...,0,0,1,2017,10,4,0,10,2017-10-02,2017-10-01
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,19.137766,...,0,0,1,2018,7,3,1,20,2018-07-24,2018-07-01
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,26.639711,...,0,0,1,2018,8,3,2,8,2018-08-08,2018-08-01
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,26.188819,...,0,0,1,2017,11,4,5,19,2017-11-18,2017-11-01
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,12.112049,...,0,0,1,2018,2,1,1,21,2018-02-13,2018-02-01


In [23]:
customer_order_features = (

    order_features

    .groupby(
        "customer_id",
        as_index=False
    )

    .agg(

        total_orders=(
            "order_id",
            "nunique"
        ),

        total_spend=(
            "total_order_value",
            "sum"
        ),

        average_order_value=(
            "total_order_value",
            "mean"
        ),

        total_items_purchased=(
            "total_items",
            "sum"
        ),

        average_review_score=(
            "review_score",
            "mean"
        ),

        average_delivery_days=(
            "delivery_days",
            "mean"
        ),

        average_delivery_delay_days=(
            "delivery_delay_days",
            "mean"
        ),

        late_orders=(
            "is_late",
            "sum"
        ),

        cancelled_orders=(
            "is_cancelled",
            "sum"
        ),

        first_purchase_date=(
            "order_purchase_timestamp",
            "min"
        ),

        last_purchase_date=(
            "order_purchase_timestamp",
            "max"
        )

    )
)

customer_order_features[
    "customer_lifetime_days"
] = (

    customer_order_features[
        "last_purchase_date"
    ]

    -

    customer_order_features[
        "first_purchase_date"
    ]

).dt.total_seconds() / 86400

customer_order_features[
    "purchase_frequency"
] = np.where(

    customer_order_features[
        "customer_lifetime_days"
    ]

    > 0,

    customer_order_features[
        "total_orders"
    ]

    /

    customer_order_features[
        "customer_lifetime_days"
    ],

    customer_order_features[
        "total_orders"
    ]

)

display(
    customer_order_features.head()
)

,customer_id,total_orders,total_spend,average_order_value,total_items_purchased,average_review_score,average_delivery_days,average_delivery_delay_days,late_orders,cancelled_orders,first_purchase_date,last_purchase_date,customer_lifetime_days,purchase_frequency
0,00012a2ce6f8dcda20d059ce98491703,1,114.74,114.74,1.0,1.0,13.981296,-5.346181,0,0,2017-11-14 16:08:26,2017-11-14 16:08:26,0.0,1.0
1,000161a058600d5901f007fab4c27140,1,67.41,67.41,1.0,4.0,9.386817,-9.210035,0,0,2017-07-16 09:40:32,2017-07-16 09:40:32,0.0,1.0
2,0001fd6190edaaf884bcaf3d49edf079,1,195.42,195.42,1.0,5.0,5.910486,-15.626516,0,0,2017-02-28 11:06:43,2017-02-28 11:06:43,0.0,1.0
3,0002414f95344307404f0ace7a26f1d5,1,179.35,179.35,1.0,5.0,28.289375,-0.162477,0,0,2017-08-16 13:09:20,2017-08-16 13:09:20,0.0,1.0
4,000379cdec625522490c315e70c7a9fb,1,107.01,107.01,1.0,4.0,11.276979,-4.151991,0,0,2018-04-02 13:42:17,2018-04-02 13:42:17,0.0,1.0


In [24]:
customer_features = (

    customers_df

    .merge(
        customer_order_features,
        on="customer_id",
        how="left"
    )
)

customer_features[
    "total_orders"
] = customer_features[
    "total_orders"
].fillna(0)

customer_features[
    "total_spend"
] = customer_features[
    "total_spend"
].fillna(0)

customer_features[
    "total_items_purchased"
] = customer_features[
    "total_items_purchased"
].fillna(0)

display(
    customer_features.head()
)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_orders,total_spend,average_order_value,total_items_purchased,average_review_score,average_delivery_days,average_delivery_delay_days,late_orders,cancelled_orders,first_purchase_date,last_purchase_date,customer_lifetime_days,purchase_frequency
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,1,146.87,146.87,1.0,4.0,8.812500,-10.558623,0,0,2017-05-16 15:05:35,2017-05-16 15:05:35,0.0,1.0
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,1,335.48,335.48,1.0,5.0,16.661748,-7.471308,0,0,2018-01-12 20:48:24,2018-01-12 20:48:24,0.0,1.0
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,1,157.73,157.73,1.0,5.0,26.077153,1.749201,1,0,2018-05-19 16:07:45,2018-05-19 16:07:45,0.0,1.0
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,1,173.30,173.30,1.0,5.0,14.998461,-12.330266,0,0,2018-03-13 16:06:38,2018-03-13 16:06:38,0.0,1.0
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,1,252.25,252.25,1.0,5.0,11.461319,-5.127917,0,0,2018-07-29 09:51:30,2018-07-29 09:51:30,0.0,1.0


In [25]:
reference_date = (

    order_features[
        "order_purchase_timestamp"
    ]

    .max()

    +

    pd.Timedelta(days=1)

)

customer_features[
    "recency_days"
] = (

    reference_date

    -

    customer_features[
        "last_purchase_date"
    ]

).dt.total_seconds() / 86400

display(
    customer_features[
        [
            "customer_id",
            "customer_unique_id",
            "total_orders",
            "total_spend",
            "recency_days"
        ]
    ].head()
)

,customer_id,customer_unique_id,total_orders,total_spend,recency_days
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,1,146.87,520.100498
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,1,335.48,278.862431
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1,157.73,152.057326
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,1,173.30,219.058102
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,1,252.25,81.318611


In [26]:
customer_features[
    "frequency"
] = customer_features[
    "total_orders"
]

customer_features[
    "monetary"
] = customer_features[
    "total_spend"
]

customer_features[
    "recency_score"
] = pd.qcut(

    customer_features[
        "recency_days"
    ]

    .rank(
        method="first"
    ),

    q=5,

    labels=[
        5,
        4,
        3,
        2,
        1
    ]

)

customer_features[
    "frequency_score"
] = pd.qcut(

    customer_features[
        "frequency"
    ]

    .rank(
        method="first"
    ),

    q=5,

    labels=[
        1,
        2,
        3,
        4,
        5
    ]

)

customer_features[
    "monetary_score"
] = pd.qcut(

    customer_features[
        "monetary"
    ]

    .rank(
        method="first"
    ),

    q=5,

    labels=[
        1,
        2,
        3,
        4,
        5
    ]

)

customer_features[
    "rfm_score"
] = (

    customer_features[
        "recency_score"
    ].astype(int).astype(str)

    +

    customer_features[
        "frequency_score"
    ].astype(int).astype(str)

    +

    customer_features[
        "monetary_score"
    ].astype(int).astype(str)

)

display(
    customer_features.head()
)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_orders,total_spend,average_order_value,total_items_purchased,average_review_score,...,last_purchase_date,customer_lifetime_days,purchase_frequency,recency_days,frequency,monetary,recency_score,frequency_score,monetary_score,rfm_score
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,1,146.87,146.87,1.0,4.0,...,2017-05-16 15:05:35,0.0,1.0,520.100498,1,146.87,1,1,4,114
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,1,335.48,335.48,1.0,5.0,...,2018-01-12 20:48:24,0.0,1.0,278.862431,1,335.48,3,1,5,315
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,1,157.73,157.73,1.0,5.0,...,2018-05-19 16:07:45,0.0,1.0,152.057326,1,157.73,4,1,4,414
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,1,173.30,173.30,1.0,5.0,...,2018-03-13 16:06:38,0.0,1.0,219.058102,1,173.30,4,1,4,414
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,1,252.25,252.25,1.0,5.0,...,2018-07-29 09:51:30,0.0,1.0,81.318611,1,252.25,5,1,5,515


In [27]:
product_sales_features = (

    order_items_df

    .groupby(
        "product_id",
        as_index=False
    )

    .agg(

        total_units_sold=(
            "order_id",
            "count"
        ),

        total_revenue=(
            "price",
            "sum"
        ),

        average_price=(
            "price",
            "mean"
        ),

        total_freight_value=(
            "freight_value",
            "sum"
        ),

        unique_orders=(
            "order_id",
            "nunique"
        ),

        unique_sellers=(
            "seller_id",
            "nunique"
        )

    )
)

product_features = (

    products_df

    .merge(
        product_sales_features,
        on="product_id",
        how="left"
    )
)

product_features[
    "total_units_sold"
] = product_features[
    "total_units_sold"
].fillna(0)

product_features[
    "total_revenue"
] = product_features[
    "total_revenue"
].fillna(0)

display(
    product_features.head()
)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,total_units_sold,total_revenue,average_price,total_freight_value,unique_orders,unique_sellers
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery,1,10.91,10.91,7.39,1,1
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art,1,248.00,248.00,17.99,1,1
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure,1,79.80,79.80,7.82,1,1
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby,1,112.30,112.30,9.54,1,1
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares,1,37.90,37.90,8.29,1,1


In [28]:
product_review_features = (

    order_items_df

    .merge(

        order_reviews_df[
            [
                "order_id",
                "review_score"
            ]
        ],

        on="order_id",

        how="left"

    )

    .groupby(
        "product_id",
        as_index=False
    )

    .agg(

        average_product_review_score=(
            "review_score",
            "mean"
        ),

        review_count=(
            "review_score",
            "count"
        )

    )
)

product_features = (

    product_features

    .merge(
        product_review_features,
        on="product_id",
        how="left"
    )
)

display(
    product_features.head()
)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,total_units_sold,total_revenue,average_price,total_freight_value,unique_orders,unique_sellers,average_product_review_score,review_count
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery,1,10.91,10.91,7.39,1,1,5.0,1
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art,1,248.00,248.00,17.99,1,1,5.0,1
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure,1,79.80,79.80,7.82,1,1,5.0,1
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby,1,112.30,112.30,9.54,1,1,1.0,1
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares,1,37.90,37.90,8.29,1,1,5.0,1


In [29]:
seller_sales_features = (

    order_items_df

    .groupby(
        "seller_id",
        as_index=False
    )

    .agg(

        total_items_sold=(
            "order_id",
            "count"
        ),

        total_revenue=(
            "price",
            "sum"
        ),

        average_item_price=(
            "price",
            "mean"
        ),

        total_freight_value=(
            "freight_value",
            "sum"
        ),

        unique_products_sold=(
            "product_id",
            "nunique"
        ),

        unique_orders=(
            "order_id",
            "nunique"
        )

    )
)

seller_features = (

    sellers_df

    .merge(
        seller_sales_features,
        on="seller_id",
        how="left"
    )
)

seller_features[
    "total_items_sold"
] = seller_features[
    "total_items_sold"
].fillna(0)

seller_features[
    "total_revenue"
] = seller_features[
    "total_revenue"
].fillna(0)

display(
    seller_features.head()
)

,seller_id,seller_zip_code_prefix,seller_city,seller_state,total_items_sold,total_revenue,average_item_price,total_freight_value,unique_products_sold,unique_orders
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,3,218.70,72.900000,27.90,3,3
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,41,11703.07,285.440732,1438.73,30,40
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,1,158.00,158.000000,16.21,1,1
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,1,79.99,79.990000,15.66,1,1
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,1,167.99,167.990000,31.93,1,1


In [30]:
monthly_sales_features = (

    order_features

    .dropna(
        subset=[
            "purchase_month_start"
        ]
    )

    .groupby(
        "purchase_month_start",
        as_index=False
    )

    .agg(

        total_orders=(
            "order_id",
            "nunique"
        ),

        total_revenue=(
            "total_order_value",
            "sum"
        ),

        total_items_sold=(
            "total_items",
            "sum"
        ),

        average_order_value=(
            "total_order_value",
            "mean"
        ),

        average_delivery_days=(
            "delivery_days",
            "mean"
        ),

        average_review_score=(
            "review_score",
            "mean"
        ),

        late_orders=(
            "is_late",
            "sum"
        ),

        cancelled_orders=(
            "is_cancelled",
            "sum"
        )

    )

)

monthly_sales_features[
    "revenue_growth"
] = (

    monthly_sales_features[
        "total_revenue"
    ]

    .pct_change()

)

monthly_sales_features[
    "order_growth"
] = (

    monthly_sales_features[
        "total_orders"
    ]

    .pct_change()

)

display(
    monthly_sales_features.head()
)

,purchase_month_start,total_orders,total_revenue,total_items_sold,average_order_value,average_delivery_days,average_review_score,late_orders,cancelled_orders,revenue_growth,order_growth
0,2016-09-01,4,354.75,6.0,118.250000,42.318715,1.000000,1,2,NaN,NaN
1,2016-10-01,324,56808.84,363.0,184.444286,25.228374,3.561129,3,24,159.137674,80.000000
2,2016-12-01,1,19.62,1.0,19.620000,4.693021,5.000000,0,0,-0.999655,-0.996914
3,2017-01-01,800,137188.49,955.0,173.876413,14.415607,4.062658,23,3,6991.277778,799.000000
4,2017-02-01,1780,286280.62,1951.0,165.193664,14.665599,4.015837,53,17,1.086769,1.225000


In [31]:
category_features = (

    products_df

    .merge(

        order_items_df[
            [
                "product_id",
                "price"
            ]
        ],

        on="product_id",

        how="left"

    )

    .groupby(
        "product_category_name",
        as_index=False
    )

    .agg(

        unique_products=(
            "product_id",
            "nunique"
        ),

        total_units_sold=(
            "product_id",
            "count"
        ),

        total_revenue=(
            "price",
            "sum"
        ),

        average_product_price=(
            "price",
            "mean"
        )

    )

)

display(
    category_features.head()
)

,product_category_name,unique_products,total_units_sold,total_revenue,average_product_price
0,agro_industria_e_comercio,74,212,72530.47,342.124858
1,alimentos,82,510,29393.41,57.634137
2,alimentos_bebidas,104,278,15179.48,54.602446
3,artes,55,209,24202.64,115.802105
4,artes_e_artesanato,19,24,1814.01,75.583750


In [32]:
feature_datasets = {

    "order_features": order_features,

    "customer_features": customer_features,

    "product_features": product_features,

    "seller_features": seller_features,

    "monthly_sales_features": monthly_sales_features,

    "category_features": category_features

}

quality_summary = []

for dataset_name, dataframe in feature_datasets.items():

    quality_summary.append({

        "dataset": dataset_name,

        "rows": dataframe.shape[0],

        "columns": dataframe.shape[1],

        "missing_values": int(
            dataframe.isna().sum().sum()
        ),

        "duplicate_rows": int(
            dataframe.duplicated().sum()
        )

    })

quality_summary_df = pd.DataFrame(
    quality_summary
)

display(
    quality_summary_df
)

,dataset,rows,columns,missing_values,duplicate_rows
0,order_features,99441,35,3871,0
1,customer_features,99441,25,1543,0
2,product_features,32951,18,162,0
3,seller_features,3095,10,0,0
4,monthly_sales_features,25,11,3,0
5,category_features,74,5,0,0


In [33]:
for dataset_name, dataframe in feature_datasets.items():

    numeric_columns = dataframe.select_dtypes(
        include=[
            "number"
        ]
    ).columns

    dataframe[numeric_columns] = (

        dataframe[numeric_columns]

        .replace(
            [
                np.inf,
                -np.inf
            ],

            np.nan
        )

    )

    dataframe[numeric_columns] = (

        dataframe[numeric_columns]

        .fillna(0)

    )

print(
    "Feature tables cleaned successfully."
)

Feature tables cleaned successfully.


In [34]:
output_files = {

    "order_features": "order_features.csv",

    "customer_features": "customer_features.csv",

    "product_features": "product_features.csv",

    "seller_features": "seller_features.csv",

    "monthly_sales_features": "monthly_sales_features.csv",

    "category_features": "category_features.csv"

}

for dataset_name, filename in output_files.items():

    output_path = os.path.join(

        FEATURE_ENGINEERED_PATH,

        filename

    )

    feature_datasets[
        dataset_name
    ].to_csv(

        output_path,

        index=False

    )

    print(
        f"Saved: {output_path}"
    )

Saved: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\feature_engineered\order_features.csv
Saved: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\feature_engineered\customer_features.csv
Saved: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\feature_engineered\product_features.csv
Saved: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\feature_engineered\seller_features.csv
Saved: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\feature_engineered\monthly_sales_features.csv
Saved: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\feature_engineered\category_features.csv


In [35]:
saved_files = os.listdir(
    FEATURE_ENGINEERED_PATH
)

print(
    "Feature-engineered files:\n"
)

for filename in saved_files:

    print(filename)

Feature-engineered files:

category_features.csv
customer_features.csv
monthly_sales_features.csv
order_features.csv
product_features.csv
seller_features.csv


In [36]:
create_schema_query = """
CREATE SCHEMA IF NOT EXISTS feature_engineered;
"""

with engine.begin() as connection:

    connection.execute(
        text(create_schema_query)
    )

print(
    "PostgreSQL feature_engineered schema is ready."
)

PostgreSQL feature_engineered schema is ready.


In [37]:
postgres_table_names = {

    "order_features": "order_features",

    "customer_features": "customer_features",

    "product_features": "product_features",

    "seller_features": "seller_features",

    "monthly_sales_features": "monthly_sales_features",

    "category_features": "category_features"

}

for dataset_name, table_name in postgres_table_names.items():

    dataframe = feature_datasets[
        dataset_name
    ]

    dataframe.to_sql(

        name=table_name,

        con=engine,

        schema="feature_engineered",

        if_exists="replace",

        index=False,

        method="multi"

    )

    print(
        f"Uploaded: feature_engineered.{table_name}"
    )

Uploaded: feature_engineered.order_features
Uploaded: feature_engineered.customer_features
Uploaded: feature_engineered.product_features
Uploaded: feature_engineered.seller_features
Uploaded: feature_engineered.monthly_sales_features
Uploaded: feature_engineered.category_features


In [38]:
feature_tables_query = """
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_schema = 'feature_engineered'
ORDER BY table_name;
"""

feature_tables_df = pd.read_sql(

    text(feature_tables_query),

    engine

)

display(
    feature_tables_df
)

,table_schema,table_name
0,feature_engineered,category_features
1,feature_engineered,customer_features
2,feature_engineered,monthly_sales_features
3,feature_engineered,order_features
4,feature_engineered,product_features
5,feature_engineered,seller_features


In [39]:
feature_row_counts = []

for table_name in postgres_table_names.values():

    query = f"""
    SELECT
        '{table_name}' AS table_name,
        COUNT(*) AS row_count
    FROM feature_engineered."{table_name}";
    """

    count_df = pd.read_sql(

        text(query),

        engine

    )

    feature_row_counts.append(
        count_df.iloc[0].to_dict()
    )

feature_row_counts_df = pd.DataFrame(
    feature_row_counts
)

display(
    feature_row_counts_df
)

,table_name,row_count
0,order_features,99441
1,customer_features,99441
2,product_features,32951
3,seller_features,3095
4,monthly_sales_features,25
5,category_features,74


In [40]:
final_summary = pd.DataFrame({

    "feature_table": [
        "order_features",
        "customer_features",
        "product_features",
        "seller_features",
        "monthly_sales_features",
        "category_features"
    ],

    "purpose": [

        "Order-level analytics and delivery analysis",

        "Customer segmentation, CLV, and churn modeling",

        "Product analytics and recommendation systems",

        "Seller performance analytics",

        "Sales forecasting and Power BI time-series dashboards",

        "Category performance analysis"

    ]

})

display(
    final_summary
)

,feature_table,purpose
0,order_features,Order-level analytics and delivery analysis
1,customer_features,"Customer segmentation, CLV, and churn modeling"
2,product_features,Product analytics and recommendation systems
3,seller_features,Seller performance analytics
4,monthly_sales_features,Sales forecasting and Power BI time-series das...
5,category_features,Category performance analysis


In [41]:
print("=" * 70)
print("FEATURE ENGINEERING COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nCSV outputs saved to:")
print(FEATURE_ENGINEERED_PATH)

print("\nPostgreSQL schema created:")
print("feature_engineered")

print("\nFeature tables created:")
print(
    list(
        postgres_table_names.values()
    )
)

print("\nReady for:")
print("- Customer Analytics")
print("- Machine Learning")
print("- Recommendation Systems")
print("- Sales Forecasting")
print("- AI Insights")
print("- Power BI Dashboards")

FEATURE ENGINEERING COMPLETED SUCCESSFULLY

CSV outputs saved to:
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\feature_engineered

PostgreSQL schema created:
feature_engineered

Feature tables created:
['order_features', 'customer_features', 'product_features', 'seller_features', 'monthly_sales_features', 'category_features']

Ready for:
- Customer Analytics
- Machine Learning
- Recommendation Systems
- Sales Forecasting
- AI Insights
- Power BI Dashboards
